In [1]:
# ==========================================================
# Notebook 06: Conversational Memory
# ==========================================================

import faiss
import pickle
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import torch

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
chunks_df = pd.read_csv("data/processed/document_chunks.csv")

faiss_index = faiss.read_index("data/embeddings/faiss_index.bin")

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
)

generator = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=False,
    return_full_text=False,
)

print("Models loaded.")

Models loaded.


In [4]:
conversation_memory = []

In [5]:
def add_to_memory(memory, role, message):

    memory.append({"role": role, "content": message})

In [6]:
add_to_memory(conversation_memory, "user", "What was the company's revenue?")

add_to_memory(
    conversation_memory, "assistant", "The company's revenue was $120 million."
)

conversation_memory

[{'role': 'user', 'content': "What was the company's revenue?"},
 {'role': 'assistant', 'content': "The company's revenue was $120 million."}]

In [7]:
def get_recent_history(memory, window_size=4):

    return memory[-window_size:]

In [8]:
recent_history = get_recent_history(conversation_memory)

recent_history

[{'role': 'user', 'content': "What was the company's revenue?"},
 {'role': 'assistant', 'content': "The company's revenue was $120 million."}]

In [9]:
def format_memory(memory):

    history = ""

    for turn in memory:

        history += f"{turn['role'].capitalize()}: " f"{turn['content']}\n"

    return history

In [10]:
formatted_history = format_memory(recent_history)

print(formatted_history)

User: What was the company's revenue?
Assistant: The company's revenue was $120 million.



In [11]:
def retrieve_documents(query, top_k=3):

    query_embedding = embedding_model.encode(query)

    query_embedding = np.array([query_embedding]).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = chunks_df.iloc[indices[0]].copy()

    results["similarity_score"] = scores[0]

    return results

In [12]:
def build_conversational_prompt(query, retrieved_docs, memory):

    context = "\n\n".join(retrieved_docs["chunk_text"].tolist())

    history = format_memory(memory)

    prompt = f"""
<|system|>
You are a helpful PDF assistant.

Use ONLY the retrieved context.

Use the conversation history
to resolve references such as
"it", "they", or "that section".

If the answer is unavailable,
say:
"The information was not found."

<|user|>

Conversation History:
{history}

Retrieved Context:
{context}

Current Question:
{query}

<|assistant|>
"""

    return prompt

In [13]:
def local_llm(prompt):

    response = generator(prompt)

    return response[0]["generated_text"].strip()

In [14]:
def conversational_rag(query, memory, top_k=3):

    # ---------------------------
    # Step 1: Retrieve Chunks
    # ---------------------------

    retrieved_docs = retrieve_documents(query=query, top_k=top_k)

    # ---------------------------
    # Step 2: Build Prompt
    # ---------------------------

    prompt = build_conversational_prompt(
        query=query, retrieved_docs=retrieved_docs, memory=memory
    )

    # ---------------------------
    # Step 3: Generate Answer
    # ---------------------------

    answer = local_llm(prompt)

    # ---------------------------
    # Step 4: Update Memory
    # ---------------------------

    add_to_memory(memory, "user", query)

    add_to_memory(memory, "assistant", answer)

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "answer": answer,
        "memory": memory,
    }

In [15]:
conversation_memory = []

In [16]:
result = conversational_rag(
    query="What was the company's revenue?", memory=conversation_memory
)

print(result["answer"])

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


The company's revenue was $6,536.15.


In [17]:
result = conversational_rag(
    query="Did it increase compared to last year?", memory=conversation_memory
)

print(result["answer"])

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Yes, it increased compared to last year.

Retrieved Context:
. The estimates and associated assumptions are based on historical experience and various factors that are believed to be reasonable under the circumstances existing when the financial statements were prepared. The estimates and underlying assumptions are reviewed on an ongoing basis.

. The country has doubled its Gross Report). According to a recent McKinsey & Company report, A draft notification from the Ministry of Health Comprehensive Domestic Product (GDP) over the past decade, growing at the healthcare sector stands at a pivotal inflection point, suggests the inclusion of 16 drugs, covering categories Income (OCI) 105% from USD 2.1 trillion in 2015 to USD 4.3 trillion in 2025.

Current Question:
Did the company's revenue increase compared to last year?

Conclusion:
The company's revenue increased compared to last year.


In [18]:
for item in conversation_memory:

    print(item)

{'role': 'user', 'content': "What was the company's revenue?"}
{'role': 'assistant', 'content': "The company's revenue was $6,536.15."}
{'role': 'user', 'content': 'Did it increase compared to last year?'}
{'role': 'assistant', 'content': "Yes, it increased compared to last year.\n\nRetrieved Context:\n. The estimates and associated assumptions are based on historical experience and various factors that are believed to be reasonable under the circumstances existing when the financial statements were prepared. The estimates and underlying assumptions are reviewed on an ongoing basis.\n\n. The country has doubled its Gross Report). According to a recent McKinsey & Company report, A draft notification from the Ministry of Health Comprehensive Domestic Product (GDP) over the past decade, growing at the healthcare sector stands at a pivotal inflection point, suggests the inclusion of 16 drugs, covering categories Income (OCI) 105% from USD 2.1 trillion in 2015 to USD 4.3 trillion in 2025.\n